# Bangla Regional Dialect Identification
**CSC 4233 NLP, Section B: Mid-Term Project** | Group G__ | Members: [Name, ID], [Name, ID], [Name, ID]

**Task:** given a short Bangla sentence, predict its regional dialect (12-class text classification).
**Data:** BanglaDial (Mahi & Khan, Mendeley Data, DOI 10.17632/sx6ybcps2n.2, CC BY 4.0).
**How to run:** `Runtime -> Run all`. No uploads or manual setup are needed; the data is read from a public GitHub link.

| Part | Content |
|---|---|
| 1 | Setup, data loading, cleaning for a fair evaluation |
| 2 | Exploratory data analysis (EDA) |
| 3 | Preprocessing pipeline (reusable function) |
| 4 | Stratified train / validation / test split |
| 5 | Preprocessing ablation, representations, classical classifiers, comparison table |
| 6 | Fine-tuned transformers (BanglaBERT, BanglishBERT, XLM-R) and the final comparison |
| 7 | Stratified 5-fold cross-validation of the classical models |

## Part 1. Setup and data

### 1.1 Install and import
Colab already contains scikit-learn, pandas, matplotlib and transformers. We only install `gensim` (Word2Vec) and `nltk` (Bengali stop-word list).
All random seeds are fixed (`SEED = 42`) so that every number in the report can be reproduced.

In [ ]:
!pip -q install gensim nltk

In [ ]:
import os, re, random, unicodedata, warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 80)

# The dataset is read from a public GitHub link (the group GitHub repository).
DATA_URL = os.environ.get(
    "BANGLADIAL_URL",
    "https://raw.githubusercontent.com/GaziArnob/Bangla-Regional-Dialect-Identification/main/colab/banglaDial.csv")
print("Reading data from:", DATA_URL)

### 1.2 Load the dataset
BanglaDial has two columns: `sentence` (Bangla text) and `label` (dialect). We apply Unicode NFC normalisation immediately.
Bangla letters can be encoded in more than one way (for example `য়` can be one character or two), so without NFC identical-looking words would be counted as different words.

In [ ]:
raw = pd.read_csv(DATA_URL)
raw["sentence"] = raw["sentence"].astype(str).map(lambda s: unicodedata.normalize("NFC", s).strip())
print("shape:", raw.shape, "| columns:", raw.columns.tolist())
print("missing values:", raw.isna().sum().to_dict())
raw.sample(8, random_state=SEED)

### 1.3 Remove duplicates and conflicting labels (before any split)
A random split of a dataset that contains repeated sentences puts the *same* sentence in both train and test, which inflates the score.
We therefore (a) drop sentences that appear under **more than one dialect label** (the label cannot be trusted), and (b) keep only one copy of each exact duplicate.

To compare sentences we build a comparison key that ignores punctuation and extra spaces.
**Important:** the key removes characters by Unicode category (`P` = punctuation, `S` = symbols). A shortcut such as the regex `[^\w\s]` is wrong for Bangla, because Python does not treat Bangla vowel signs (া, ি, ে ...) as word characters and would delete them.

In [ ]:
def norm_key(s):
    s = "".join(" " if unicodedata.category(c)[0] in "PS" else c for c in s)
    return re.sub(r"\s+", " ", s).strip()

raw["key"] = raw["sentence"].map(norm_key)
n_start = len(raw)
raw = raw[raw["key"] != ""]
n_empty = n_start - len(raw)

labels_per_key = raw.groupby("key")["label"].nunique()
conflict_keys = set(labels_per_key[labels_per_key > 1].index)
n_conflict_texts = len(conflict_keys)
n_conflict_rows = raw["key"].isin(conflict_keys).sum()
raw = raw[~raw["key"].isin(conflict_keys)]

n_before_dup = len(raw)
df = raw.drop_duplicates("key").reset_index(drop=True)
n_dup = n_before_dup - len(df)

print(f"rows at start                          : {n_start}")
print(f"empty after cleaning                   : {n_empty}")
print(f"sentences with >1 label (removed)      : {n_conflict_texts} sentences = {n_conflict_rows} rows")
print(f"repeated copies (removed, 1 copy kept) : {n_dup}")
print(f"rows used for the experiments          : {len(df)}")

## Part 2. Exploratory data analysis (EDA)
We look at the class distribution (is the data imbalanced?), the sentence length, and the most frequent words per class before and after preprocessing.

### 2.1 Class distribution

In [ ]:
counts = df["label"].value_counts()
ratio = counts.max() / counts.min()
print(counts.to_string())
print(f"\nlargest / smallest class = {counts.index[0]} / {counts.index[-1]} = {ratio:.1f} : 1  -> the data is imbalanced")

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(x=counts.values, y=counts.index, color="#3b6ea5", ax=ax)
ax.set_xlabel("number of sentences"); ax.set_ylabel("dialect"); ax.set_title("Figure 1. Class distribution")
for i, v in enumerate(counts.values):
    ax.text(v + 60, i, str(v), va="center", fontsize=8)
plt.tight_layout(); plt.show()

**Figure 1 interpretation:** *(write one line after you look at the figure: which classes are largest/smallest, and why this makes macro-F1 the right main metric)*

### 2.2 Sentence length

In [ ]:
df["n_words"] = df["sentence"].str.split().str.len()
df["n_chars"] = df["sentence"].str.len()
print(df[["n_words", "n_chars"]].describe().round(1).to_string())

fig, ax = plt.subplots(figsize=(8, 3.6))
sns.histplot(df["n_words"], bins=range(1, 31), color="#3b6ea5", ax=ax)
ax.set_xlabel("words per sentence"); ax.set_title("Figure 2. Sentence length (words)")
plt.tight_layout(); plt.show()

**Figure 2 interpretation:** *(one line: typical sentence length, and what very short sentences mean for n-gram / embedding models)*

### 2.3 Sentence length per dialect

In [ ]:
order = df.groupby("label")["n_words"].median().sort_values().index
fig, ax = plt.subplots(figsize=(9, 4.2))
sns.boxplot(data=df, y="label", x="n_words", order=order, color="#9dc0e0", showfliers=False, ax=ax)
ax.set_xlabel("words per sentence"); ax.set_ylabel("dialect"); ax.set_title("Figure 3. Sentence length by dialect")
plt.tight_layout(); plt.show()

**Figure 3 interpretation:** *(one line: do some dialects have systematically shorter or longer sentences? If yes, length could act as a shortcut and must be noted)*

## Part 3. Preprocessing pipeline
One reusable function, `preprocess(text, ...)`. Every step is switchable, so the ablation study (later) can turn steps on and off.

| Step | What it does | Why |
|---|---|---|
| Unicode NFC | canonical form of every character | identical-looking words must be identical strings |
| Regex cleaning | removes URLs, HTML tags, emoji/symbols, Latin letters | BanglaDial is already clean, so this step is a safety net (we measure how often it fires) |
| Digits | optional: replace Bangla/ASCII digits with the token `NUM` | numbers carry no dialect information but may carry noise |
| Bangla tokeniser | keeps maximal runs of Bengali-block characters `[\u0980-\u09FF]+` | unlike `str.split()` or `\w+`, it never cuts a word at a vowel sign |
| Stop-word removal | optional: NLTK Bengali stop-word list (about 390 words) | removes frequent function words |
| Stemming | optional: simple suffix stripper (rule based) | merges inflected forms |

*Case folding is not needed: Bangla has no upper/lower case and the corpus contains almost no English.*
Punctuation is already removed in the source data, so punctuation removal cannot be ablated; this is a limitation we report.

In [ ]:
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

BN_STOP = {unicodedata.normalize("NFC", w) for w in stopwords.words("bengali")}
print("Bengali stop-words:", len(BN_STOP), "| examples:", sorted(BN_STOP)[:10])

URL_RE   = re.compile(r"(https?://\S+|www\.\S+)")
HTML_RE  = re.compile(r"<[^>]+>|&\w+;")
DIGIT_RE = re.compile(r"[0-9\u09E6-\u09EF]+")            # ASCII and Bangla digits
BN_TOKEN = re.compile(r"[\u0980-\u09FF]+")               # Bengali Unicode block

# A deliberately simple suffix list (longest first). Real Bangla morphology is much richer, see limitations below.
SUFFIXES = sorted(["গুলোকে", "গুলোর", "গুলির", "গুলো", "গুলি", "গুলা", "দেরকে", "দের", "েরা", "ের",
                   "কে", "রে", "তে", "টি", "টা", "টো", "খানা", "খানি", "রা", "ে", "র"], key=len, reverse=True)

def stem_word(w, min_len=2):
    for suf in SUFFIXES:
        if w.endswith(suf) and len(w) - len(suf) >= min_len:
            return w[: -len(suf)]
    return w

def tokenize(text, remove_stopwords=False, stem=False, mask_digits=False):
    t = unicodedata.normalize("NFC", str(text))
    t = URL_RE.sub(" ", t)
    t = HTML_RE.sub(" ", t)
    t = DIGIT_RE.sub(" NUM " if mask_digits else " ", t)
    out = []
    for chunk in t.split():
        if chunk == "NUM":
            out.append("NUM")
        else:
            out += BN_TOKEN.findall(chunk)
    if remove_stopwords:
        out = [w for w in out if w not in BN_STOP]
    if stem:
        out = [w if w == "NUM" else stem_word(w) for w in out]
    return out

def preprocess(text, **kw):
    return " ".join(tokenize(text, **kw))

demo = df["sentence"].sample(5, random_state=3).tolist() + ["আমি ১২৫ টাকা দিছি http://x.com <b>ভালো</b> 😀 hello"]
for s in demo:
    print("raw      :", s)
    print("tokens   :", preprocess(s))
    print("-stop    :", preprocess(s, remove_stopwords=True))
    print("-stop+stem:", preprocess(s, remove_stopwords=True, stem=True))
    print("digits   :", preprocess(s, mask_digits=True))
    print()

### 3.1 How often do the cleaning rules actually fire?
The assignment asks for regex cleaning of URLs, HTML, emoji and numbers. We count how many sentences each rule touches, so the report can say honestly whether the step matters for this corpus.

In [ ]:
EMOJI_LIKE = re.compile(r"[^\u0980-\u09FF\s0-9A-Za-z]")
has = lambda rx: df["sentence"].map(lambda t: bool(rx.search(t)))
rules = {
    "URL": has(URL_RE),
    "HTML tag / entity": has(HTML_RE),
    "digits (ASCII or Bangla)": has(DIGIT_RE),
    "Latin letters": has(re.compile(r"[A-Za-z]")),
    "emoji / other symbols": has(EMOJI_LIKE),
}
pd.DataFrame({"sentences affected": {k: int(v.sum()) for k, v in rules.items()}}).assign(
    percent=lambda t: (t["sentences affected"] / len(df) * 100).round(3))

### 3.2 Most frequent words per class, before and after preprocessing
*Before* = plain whitespace tokens. *After* = tokeniser + stop-word removal + stemming.

In [ ]:
def top_words(tokenizer, k=7):
    rows = {}
    for lab, g in df.groupby("label"):
        c = Counter(w for s in g["sentence"] for w in tokenizer(s))
        rows[lab] = " | ".join(w for w, _ in c.most_common(k))
    return pd.Series(rows)

tw = pd.DataFrame({
    "before (raw tokens)": top_words(str.split),
    "after (stop-words removed + stemmed)": top_words(lambda s: tokenize(s, remove_stopwords=True, stem=True)),
})
tw

In [ ]:
vocab_raw = len({w for s in df["sentence"] for w in s.split()})
vocab_pre = len({w for s in df["sentence"] for w in tokenize(s, remove_stopwords=True, stem=True)})
print(f"vocabulary size  raw tokens: {vocab_raw}   after preprocessing: {vocab_pre}   (-{(1 - vocab_pre / vocab_raw) * 100:.1f}%)")

**Limitations of Bangla stemming (for the report):** our stemmer only strips a short list of common suffixes. It cannot handle irregular verb forms, and **dialect verb endings differ from standard Bangla** (for example Chittagonian and Sylheti verb forms), so it may merge forms that are actually dialect markers or fail to merge real inflections. The ablation in Part 5 measures whether stemming and stop-word removal help or hurt dialect identification.

## Part 4. Train / validation / test split
Stratified **70 / 15 / 15** split with a fixed seed. Stratification keeps the class proportions identical in all three parts, which matters for the small classes (for example Rajshahi).
The test set is **not touched** until the final evaluation. Every vectoriser, vocabulary, Word2Vec model and hyper-parameter is fitted on the training part (and tuned on validation) only.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)
val_df, test_df   = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)

print(f"train {len(train_df)} | validation {len(val_df)} | test {len(test_df)}")
split_table = pd.DataFrame({
    "train": train_df["label"].value_counts(), "validation": val_df["label"].value_counts(), "test": test_df["label"].value_counts()})
split_table["train %"] = (split_table["train"] / split_table["train"].sum() * 100).round(2)
split_table["test %"]  = (split_table["test"]  / split_table["test"].sum()  * 100).round(2)
split_table.sort_values("train", ascending=False)

## Part 5. Representations and classical classifiers

The plan of this part:

| Step | What | Data used |
|---|---|---|
| 5.1 | Preprocessing **ablation**: which pipeline variant gives the best macro-F1? | validation only |
| 5.2 | Fix the pipeline for all later experiments | |
| 5.3 | Build the representations (BoW, TF-IDF, averaged Word2Vec) | fitted on **train** only |
| 5.4 | Train and tune the classifiers | tune on **validation** |
| 5.5 | Final comparison table | **test**, each final model evaluated once |

**Metric.** Because the classes are imbalanced (Figure 1), **macro-F1** (the unweighted mean of the 12 per-class F1 scores) is the main metric. Accuracy is reported as supplementary information.

In [ ]:
import time
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score

y_train, y_val, y_test = train_df["label"].values, val_df["label"].values, test_df["label"].values

def metrics(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(y_true, y_pred), "macro_P": p, "macro_R": r, "macro_F1": f}

def prep_series(series, **kw):
    return series.map(lambda t: preprocess(t, **kw))

### 5.1 Preprocessing ablation
We compare five pipeline variants. Everything else is held fixed (TF-IDF with unigrams + bigrams, fixed hyper-parameters, same split), so any change in macro-F1 comes from the preprocessing only. Two different classifiers are used so the conclusion does not depend on one model.
The comparison uses the **validation** set; the test set stays untouched.

| Variant | Meaning |
|---|---|
| A | Bangla tokeniser only (baseline) |
| B | A + stop-word removal |
| C | A + stemming |
| D | A + stop-word removal + stemming |
| E | A, with digits kept as the token `NUM` instead of being deleted |

In [ ]:
VARIANTS = {
    "A tokens only":              dict(),
    "B + stop-words removed":     dict(remove_stopwords=True),
    "C + stemming":               dict(stem=True),
    "D + stop-words + stemming":  dict(remove_stopwords=True, stem=True),
    "E digits kept as NUM":       dict(mask_digits=True),
}
rows = []
t0 = time.time()
for name, kw in VARIANTS.items():
    tr, va = prep_series(train_df["sentence"], **kw), prep_series(val_df["sentence"], **kw)
    vec = TfidfVectorizer(token_pattern=r"\S+", lowercase=False, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    Xtr, Xva = vec.fit_transform(tr), vec.transform(va)
    f_nb  = f1_score(y_val, MultinomialNB(alpha=0.1).fit(Xtr, y_train).predict(Xva), average="macro")
    f_svm = f1_score(y_val, LinearSVC(C=0.5, class_weight="balanced", random_state=SEED).fit(Xtr, y_train).predict(Xva), average="macro")
    rows.append({"variant": name, "vocabulary": Xtr.shape[1], "NB macro-F1": f_nb, "SVM macro-F1": f_svm})
abl = pd.DataFrame(rows).set_index("variant")
abl["mean macro-F1"] = abl[["NB macro-F1", "SVM macro-F1"]].mean(axis=1)
abl["change vs A"] = abl["mean macro-F1"] - abl.loc["A tokens only", "mean macro-F1"]
print(f"ablation took {time.time() - t0:.0f} s")
abl.round(4)

In [ ]:
cols = ["NB macro-F1", "SVM macro-F1"]
fig, ax = plt.subplots(figsize=(8, 3.6))
abl[cols].plot.barh(ax=ax, color=["#9dc0e0", "#3b6ea5"])
ax.set_xlim(abl[cols].min().min() - 0.03, abl[cols].max().max() + 0.01)
ax.invert_yaxis(); ax.legend(loc="lower right"); ax.set_xlabel("macro-F1 (validation)"); ax.set_title("Figure 4. Preprocessing ablation")
plt.tight_layout(); plt.show()

**Ablation interpretation:** *(write after you see the table: which variant is best, did stop-word removal or stemming help or hurt, and why this is plausible for dialects, where function words and verb endings carry the dialect signal)*

### 5.2 Fix the pipeline
The variant with the highest mean validation macro-F1 is used for all following experiments. The choice is made on validation data, not on the test set.

In [ ]:
BEST_VARIANT = abl["mean macro-F1"].idxmax()
PREP_KW = VARIANTS[BEST_VARIANT]
print("selected variant:", BEST_VARIANT, "->", PREP_KW)

txt_tr = prep_series(train_df["sentence"], **PREP_KW)
txt_va = prep_series(val_df["sentence"],   **PREP_KW)
txt_te = prep_series(test_df["sentence"],  **PREP_KW)
print(len(txt_tr), len(txt_va), len(txt_te), "| empty sentences after preprocessing (train):", int((txt_tr.str.len() == 0).sum()))

### 5.3 Representations (all fitted on the training set only)

| Representation | Setting | Why |
|---|---|---|
| **BoW** | word counts, words seen at least twice | simplest representation; required baseline |
| **TF-IDF word (1,1)** | unigrams | down-weights very common words |
| **TF-IDF word (1,2)** | unigrams + bigrams | adds a little word order (n-gram setting required by the assignment) |
| **TF-IDF char (1,4)** | character 1-4-grams inside words | dialects differ in spelling and endings; character n-grams see these pieces and tolerate unseen spellings |
| **Word2Vec average** | 100-dim skip-gram vectors trained on our training sentences, averaged per sentence | dense embedding representation (required) |

`min_df=2` removes words that occur only once: they cannot help a classifier and they inflate the vocabulary.

In [ ]:
reps = {}
def add_rep(name, vec):
    Xtr = vec.fit_transform(txt_tr)
    reps[name] = (Xtr, vec.transform(txt_va), vec.transform(txt_te))
    print(f"{name:20s} features: {Xtr.shape[1]:>7}")

add_rep("BoW",                CountVectorizer(token_pattern=r"\S+", lowercase=False, min_df=2))
add_rep("TF-IDF word (1,1)",  TfidfVectorizer(token_pattern=r"\S+", lowercase=False, min_df=2, sublinear_tf=True))
add_rep("TF-IDF word (1,2)",  TfidfVectorizer(token_pattern=r"\S+", lowercase=False, ngram_range=(1, 2), min_df=2, sublinear_tf=True))
add_rep("TF-IDF char (1,4)",  TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 4), min_df=2, sublinear_tf=True, max_features=200000))

In [ ]:
from gensim.models import Word2Vec

tok_tr = [t.split() for t in txt_tr]
w2v = Word2Vec(tok_tr, vector_size=100, window=3, min_count=2, sg=1, epochs=15, workers=1, seed=SEED)
print("Word2Vec vocabulary:", len(w2v.wv))

def sent_vec(text):
    vs = [w2v.wv[w] for w in text.split() if w in w2v.wv]
    return np.mean(vs, axis=0) if vs else np.zeros(w2v.vector_size)

W = {k: np.vstack([sent_vec(t) for t in v]) for k, v in {"tr": txt_tr, "va": txt_va, "te": txt_te}.items()}
reps["Word2Vec avg"] = (W["tr"], W["va"], W["te"])
print("sentence vectors:", W["tr"].shape, "| validation sentences with no known word:", int((np.abs(W["va"]).sum(axis=1) == 0).sum()))
for w in [w for w in ["ভালা", "মুই"] if w in w2v.wv]:
    print(w, "->", [x for x, _ in w2v.wv.most_similar(w, topn=5)])

### 5.4 Classifiers and hyper-parameter tuning
| Classifier | Used on | Tuned hyper-parameter (on validation) | Why |
|---|---|---|---|
| Majority class | - | - | the floor any real model must beat |
| **Multinomial Naive Bayes** | BoW and the TF-IDF variants | smoothing `alpha` in {0.05, 0.2, 1.0} | classic, fast text baseline; needs non-negative features, so it cannot use Word2Vec |
| **Logistic Regression** | all representations | `C` in {1, 10} | strong linear model |
| **Linear SVM** | all representations | `C` in {0.1, 1} | usually the best linear model for sparse, high-dimensional text |
| **Gaussian Naive Bayes** | Word2Vec | `var_smoothing` in {1e-9, 1e-6} | the Naive Bayes variant that accepts negative, dense features |

`class_weight="balanced"` is used for Logistic Regression and the SVM so that the small classes (Rajshahi: 885 sentences) are not ignored. Naive Bayes keeps its default class prior.
The **validation** set picks the best value in each grid; the chosen model is then evaluated once on the **test** set.

In [ ]:
results, models = [], {}

def run(rep, clf_name, make, grid):
    Xtr, Xva, Xte = reps[rep]
    best = None
    t0 = time.time()
    for p in grid:
        clf = make(p).fit(Xtr, y_train)
        f1 = f1_score(y_val, clf.predict(Xva), average="macro")
        if best is None or f1 > best[0]:
            best = (f1, p, clf)
    f1, p, clf = best
    m = metrics(y_test, clf.predict(Xte))          # the test set is used once, for the selected model only
    results.append({"representation": rep, "classifier": clf_name, "best setting": p, "val macro-F1": f1, **m})
    models[(rep, clf_name)] = clf
    print(f"{rep:20s} {clf_name:22s} best={p!s:8s} val F1={f1:.4f}  test F1={m['macro_F1']:.4f}  ({time.time() - t0:.0f}s)")

# majority-class baseline
maj = DummyClassifier(strategy="most_frequent").fit(reps["BoW"][0], y_train)
results.append({"representation": "-", "classifier": "Majority class", "best setting": "-",
                "val macro-F1": f1_score(y_val, maj.predict(reps["BoW"][1]), average="macro"), **metrics(y_test, maj.predict(reps["BoW"][2]))})

sparse_reps = ["BoW", "TF-IDF word (1,1)", "TF-IDF word (1,2)", "TF-IDF char (1,4)"]
for rep in sparse_reps:
    run(rep, "Multinomial NB", lambda a: MultinomialNB(alpha=a), [0.05, 0.2, 1.0])
for rep in sparse_reps:
    run(rep, "Linear SVM", lambda c: LinearSVC(C=c, class_weight="balanced", random_state=SEED), [0.1, 1.0])

In [ ]:
for rep in sparse_reps:
    run(rep, "Logistic Regression", lambda c: LogisticRegression(C=c, max_iter=200, class_weight="balanced"), [1.0, 10.0])
run("Word2Vec avg", "Logistic Regression", lambda c: LogisticRegression(C=c, max_iter=300, class_weight="balanced"), [1.0, 10.0])
run("Word2Vec avg", "Linear SVM",          lambda c: LinearSVC(C=c, class_weight="balanced", random_state=SEED), [0.1, 1.0])
run("Word2Vec avg", "Gaussian NB",         lambda v: GaussianNB(var_smoothing=v), [1e-9, 1e-6])

### 5.5 Comparison table (test set)
Rows are *representation + classifier*; the columns are the metrics required by the assignment. The table is sorted by **macro-F1**, the primary metric. `val macro-F1` is the score that was used to choose the hyper-parameter.

In [ ]:
res = pd.DataFrame(results).sort_values("macro_F1", ascending=False).reset_index(drop=True)
res.index += 1
res = res.rename(columns={"macro_F1": "test macro-F1", "macro_P": "macro-P", "macro_R": "macro-R"})
show = res[["representation", "classifier", "best setting", "val macro-F1", "accuracy", "macro-P", "macro-R", "test macro-F1"]]
show.style.format({c: "{:.4f}" for c in ["val macro-F1", "accuracy", "macro-P", "macro-R", "test macro-F1"]}).background_gradient(subset=["test macro-F1"], cmap="Blues")

In [ ]:
plot = res.copy()
plot["model"] = plot["representation"] + " + " + plot["classifier"]
plot = plot.sort_values("test macro-F1")
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(plot["model"], plot["test macro-F1"], color="#3b6ea5")
ax.set_xlabel("macro-F1 (test)"); ax.set_title("Figure 5. All classical models, test macro-F1")
for i, v in enumerate(plot["test macro-F1"]):
    ax.text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=7)
plt.tight_layout(); plt.show()
best = res.iloc[0]
print(f"best classical model: {best['representation']} + {best['classifier']} | macro-F1 = {best['test macro-F1']:.4f}, accuracy = {best['accuracy']:.4f}")

**Comparison interpretation:** *(write after you see the table: which representation is best and why; whether character n-grams beat word n-grams; how Word2Vec averages compare with TF-IDF; how much the best model improves over the majority baseline)*

## Part 6. Pre-trained transformer models

> **Scope note.** The course guideline limits the mid-term project to Weeks 1-5 techniques. This part uses pre-trained transformers and is included **with the instructor's approval** (confirmation: *[write date / how it was approved]*). If you do not have this approval, skip Part 6 (set `RUN_TRANSFORMERS = False` in the next cell) and report only Parts 1-5.

We fine-tune three pre-trained models end-to-end for the same 12-class task, on the **same split**, with the **same metric** (macro-F1), so they can be placed in the same comparison table as the classical models.

| Model | What it is | Why it is included |
|---|---|---|
| **BanglaBERT** (`csebuetnlp/banglabert`) | ELECTRA discriminator (about 110 M parameters), pre-trained on Bangla text | language-specific model: knows Bangla spelling and morphology |
| **BanglishBERT** (`csebuetnlp/banglishbert`) | same architecture, pre-trained on **Bangla and English** text | tests whether the extra English pre-training helps or hurts; our data is written in Bangla script |
| **XLM-R base** (`xlm-roberta-base`) | multilingual RoBERTa covering about 100 languages | general multilingual baseline that was not built for Bangla |

Check the model cards on Hugging Face for each model's licence before reusing the weights.

**Training recipe (same for all models).** AdamW optimiser, linear learning-rate decay with 10% warm-up, batch size 32, at most 4 epochs, maximum length 64 tokens (the sentences are short: 6 words on average), mixed precision on GPU, gradient clipping at 1.0. The loss is **class-weighted** cross-entropy (the same idea as `class_weight="balanced"` for the classical models). After every epoch the model is scored on the **validation** set and the weights of the best epoch are kept: this is how the number of epochs is tuned. The test set is used once, for that best epoch.

In [ ]:
RUN_TRANSFORMERS = os.environ.get("NB_SKIP_TF") != "1"     # set to False to skip Part 6 (for example if there is no GPU)
SMOKE = os.environ.get("NB_SMOKE") == "1"   # tiny quick-test mode used only while developing the notebook

import torch, gc
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.preprocessing import LabelEncoder

DEVICE = os.environ.get("NB_DEVICE") or ("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE == "cuda"
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU")
if DEVICE != "cuda" and not SMOKE:
    print("No GPU found: fine-tuning would take hours on a CPU, so Part 6 is skipped.")
    print("To use it: Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all.")
    RUN_TRANSFORMERS = False

In [ ]:
!pip -q install git+https://github.com/csebuetnlp/normalizer

In [ ]:
try:
    from normalizer import normalize          # the text normaliser used when BanglaBERT / BanglishBERT were pre-trained
    print("csebuetnlp normalizer loaded:", normalize("কেমন  আছো ?"))
except Exception as e:
    print("normalizer not available, using plain NFC text instead:", e)
    normalize = lambda s: unicodedata.normalize("NFC", s)

### 6.1 Setup: labels, data loaders, training and prediction functions
BanglaBERT and BanglishBERT were pre-trained on text that went through the `normalizer` package, so the input is passed through it first (recommended on their model cards). XLM-R receives the plain sentence.

In [ ]:
MODELS = {                                    # name: (Hugging Face id, apply csebuetnlp normaliser?)
    "BanglaBERT":   ("csebuetnlp/banglabert",   True),
    "BanglishBERT": ("csebuetnlp/banglishbert", True),
    "XLM-R base":   ("xlm-roberta-base",        False),
}
if os.environ.get("NB_MODELS"):
    MODELS = {k: v for k, v in MODELS.items() if k in os.environ["NB_MODELS"].split(",")}
MAX_LEN, BATCH = 64, 32
EPOCHS    = 1 if SMOKE else 4
LR_GRID   = [3e-5]                            # add a second value, e.g. [2e-5, 5e-5], for a real learning-rate search (doubles the time)

le = LabelEncoder().fit(df["label"])
CLASSES = list(le.classes_)
ytr_id, yva_id, yte_id = le.transform(y_train), le.transform(y_val), le.transform(y_test)
cls_count = np.bincount(ytr_id, minlength=len(CLASSES))
CLASS_W = torch.tensor(len(ytr_id) / (len(CLASSES) * cls_count), dtype=torch.float)   # same formula as sklearn's "balanced"

sub = (lambda s, n: s.iloc[:n]) if SMOKE else (lambda s, n: s)
S_TR, S_VA, S_TE = (sub(train_df["sentence"], 2000), sub(val_df["sentence"], 600), sub(test_df["sentence"], 600))
Y_TR, Y_VA, Y_TE = ytr_id[:len(S_TR)], yva_id[:len(S_VA)], yte_id[:len(S_TE)]

class EncDS(torch.utils.data.Dataset):
    def __init__(self, enc, y): self.enc, self.y = enc, y
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = int(self.y[i])
        return item

def make_loader(texts, y, tok, shuffle, bs=BATCH):
    ds = EncDS(tok(list(texts), truncation=True, max_length=MAX_LEN), y)
    def collate(batch):
        labels = torch.tensor([b.pop("labels") for b in batch])
        out = tok.pad(batch, return_tensors="pt")
        out["labels"] = labels
        return out
    return torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=shuffle, collate_fn=collate)

@torch.no_grad()
def predict_ids(model, loader):
    model.eval(); out = []
    for b in loader:
        b = {k: v.to(DEVICE) for k, v in b.items() if k != "labels"}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            out.append(model(**b).logits.argmax(-1).cpu())
    return torch.cat(out).numpy()

In [ ]:
def fine_tune(name, hub_id, use_norm, lr):
    torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(hub_id)
    prep = (lambda s: s.map(normalize)) if use_norm else (lambda s: s)
    ld_tr = make_loader(prep(S_TR), Y_TR, tok, True)
    ld_va = make_loader(prep(S_VA), Y_VA, tok, False, 64)
    ld_te = make_loader(prep(S_TE), Y_TE, tok, False, 64)

    model = AutoModelForSequenceClassification.from_pretrained(hub_id, num_labels=len(CLASSES)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total = len(ld_tr) * EPOCHS
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    loss_fn = torch.nn.CrossEntropyLoss(weight=CLASS_W.to(DEVICE))

    best = {"f1": -1, "epoch": 0, "state": None}
    history = []
    for ep in range(1, EPOCHS + 1):
        model.train(); run_loss = 0.0
        for b in ld_tr:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            labels = b.pop("labels")
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                loss = loss_fn(model(**b).logits.float(), labels)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run_loss += loss.item()
        val_f1 = f1_score(Y_VA, predict_ids(model, ld_va), average="macro")
        history.append({"epoch": ep, "train loss": run_loss / len(ld_tr), "val macro-F1": val_f1})
        print(f"{name:13s} lr={lr:g} epoch {ep}/{EPOCHS}  train loss {run_loss / len(ld_tr):.4f}  val macro-F1 {val_f1:.4f}  ({time.time() - t0:.0f}s)", flush=True)
        if val_f1 > best["f1"]:
            best.update(f1=val_f1, epoch=ep, state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
    model.load_state_dict(best["state"])
    pred = predict_ids(model, ld_te)                  # test set: used once, with the best-epoch weights
    del model, opt, best["state"]; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return {"val_f1": best["f1"], "epoch": best["epoch"], "pred": pred, "history": history, "seconds": time.time() - t0}

### 6.2 Fine-tune the three models
Each model is trained from its pre-trained weights. If `LR_GRID` has more than one value, the learning rate with the best validation macro-F1 is kept.

In [ ]:
test_preds, histories = {}, {}
if RUN_TRANSFORMERS:
    for name, (hub_id, use_norm) in MODELS.items():
        runs = [(lr, fine_tune(name, hub_id, use_norm, lr)) for lr in LR_GRID]
        lr, r = max(runs, key=lambda x: x[1]["val_f1"])
        m = metrics(Y_TE, r["pred"])
        results.append({"representation": "fine-tuned transformer", "classifier": name, "best setting": f"lr={lr:g}, epoch {r['epoch']}",
                        "val macro-F1": r["val_f1"], **m})
        test_preds[name], histories[name] = r["pred"], pd.DataFrame(r["history"])
        print(f"==> {name}: best epoch {r['epoch']}, val macro-F1 {r['val_f1']:.4f}, test macro-F1 {m['macro_F1']:.4f}, accuracy {m['accuracy']:.4f}  ({r['seconds'] / 60:.1f} min)\n")
else:
    print("Part 6 skipped.")

In [ ]:
if histories:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    for name, h in histories.items():
        axes[0].plot(h["epoch"], h["train loss"], marker="o", label=name)
        axes[1].plot(h["epoch"], h["val macro-F1"], marker="o", label=name)
    axes[0].set_title("Training loss"); axes[1].set_title("Validation macro-F1")
    for a in axes: a.set_xlabel("epoch"); a.set_xticks(sorted(set(sum([list(h["epoch"]) for h in histories.values()], []))))
    axes[1].legend(); fig.suptitle("Figure 6. Fine-tuning curves"); plt.tight_layout(); plt.show()

**Training-curve interpretation:** *(write after you see the figure: does validation macro-F1 still rise at the last epoch or does it peak earlier, and which model learns fastest?)*

### 6.3 All models in one table (test set)
Classical models from Part 5 and the fine-tuned transformers, sorted by macro-F1 (primary metric).

In [ ]:
res_all = pd.DataFrame(results).sort_values("macro_F1", ascending=False).reset_index(drop=True)
res_all.index += 1
res_all = res_all.rename(columns={"macro_F1": "test macro-F1", "macro_P": "macro-P", "macro_R": "macro-R"})
cols = ["representation", "classifier", "best setting", "val macro-F1", "accuracy", "macro-P", "macro-R", "test macro-F1"]
res_all[cols].style.format({c: "{:.4f}" for c in cols[3:]}).background_gradient(subset=["test macro-F1"], cmap="Blues")

In [ ]:
plot = res_all.copy()
plot["model"] = plot["representation"].replace({"-": ""}) + " " + plot["classifier"]
plot = plot.sort_values("test macro-F1")
colors = ["#d9822b" if r == "fine-tuned transformer" else "#3b6ea5" for r in plot["representation"]]
fig, ax = plt.subplots(figsize=(9, 6.5))
ax.barh(plot["model"].str.strip(), plot["test macro-F1"], color=colors)
for i, v in enumerate(plot["test macro-F1"]):
    ax.text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=7)
ax.set_xlabel("macro-F1 (test)"); ax.set_title("Figure 7. All models (orange = fine-tuned transformers)")
plt.tight_layout(); plt.show()
top = res_all.iloc[0]
print(f"best model overall: {top['representation']} {top['classifier']} | macro-F1 = {top['test macro-F1']:.4f}, accuracy = {top['accuracy']:.4f}")

**Transformer interpretation:** *(write after you see the table: do the transformers beat the best classical model (TF-IDF char + Linear SVM)? Which transformer is best, and does English pre-training (BanglishBERT) help? Compare the cost (training time) with the gain.)*

## Part 7. Stratified 5-fold cross-validation (classical models)

So far each model was scored on **one** test set of 8,982 sentences. A single split can be lucky or unlucky, and it leaves 30% of the sentences out of training. **Stratified 5-fold cross-validation** uses *all* 59,880 sentences:

1. The data is cut into 5 folds that keep the class proportions (Rajshahi is about 1.5% in every fold).
2. In round *k*, fold *k* is the test set and the other 4 folds (about 47,900 sentences) are the training set.
3. After 5 rounds every sentence has been predicted exactly once by a model that never saw it. We report the **mean and standard deviation** of the 5 scores.

**Rules kept inside each round:** the vectoriser (and the Word2Vec model) is fitted on the 4 training folds only; the test fold is touched once.
**Hyper-parameters** are the ones selected on the validation set in Part 5 (the grid search is not repeated in every fold, to keep the run time reasonable). Because the Part 5 validation sentences are also part of some test folds, this gives a slightly optimistic estimate; a fully clean estimate would tune inside every fold (nested cross-validation). We note this as a limitation.

The fine-tuned transformers of Part 6 stay on the single train / validation / test split: fine-tuning one model five times would multiply the training time by five. Their scores are therefore **not strictly comparable** with the cross-validated means below, and the report should say so.

In [ ]:
from sklearn.model_selection import StratifiedKFold

y_all   = df["label"].values
txt_all = prep_series(df["sentence"], **PREP_KW)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

VEC = {
    "BoW":               lambda: CountVectorizer(token_pattern=r"\S+", lowercase=False, min_df=2),
    "TF-IDF word (1,1)": lambda: TfidfVectorizer(token_pattern=r"\S+", lowercase=False, min_df=2, sublinear_tf=True),
    "TF-IDF word (1,2)": lambda: TfidfVectorizer(token_pattern=r"\S+", lowercase=False, ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    "TF-IDF char (1,4)": lambda: TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 4), min_df=2, sublinear_tf=True, max_features=200000),
}
CLF = {
    "Multinomial NB":      lambda p, rep: MultinomialNB(alpha=p),
    "Linear SVM":          lambda p, rep: LinearSVC(C=p, class_weight="balanced", random_state=SEED),
    "Logistic Regression": lambda p, rep: LogisticRegression(C=p, max_iter=300 if rep == "Word2Vec avg" else 200, class_weight="balanced"),
    "Gaussian NB":         lambda p, rep: GaussianNB(var_smoothing=p),
}
# hyper-parameters chosen on the validation set in Part 5 (classical models only)
cfg = [(r["representation"], r["classifier"], r["best setting"]) for r in results
       if r["representation"] not in ("-", "fine-tuned transformer")]
print(len(cfg), "classical configurations will be cross-validated")

def w2v_features(tr_texts, te_texts):
    w = Word2Vec([t.split() for t in tr_texts], vector_size=100, window=3, min_count=2, sg=1, epochs=15, workers=1, seed=SEED)
    vec = lambda t: (np.mean([w.wv[x] for x in t.split() if x in w.wv], axis=0) if any(x in w.wv for x in t.split()) else np.zeros(100))
    return np.vstack([vec(t) for t in tr_texts]), np.vstack([vec(t) for t in te_texts])

In [ ]:
fold_scores = {(r, c): [] for r, c, _ in cfg}
fold_scores[("-", "Majority class")] = []
oof = {k: np.empty(len(df), dtype=object) for k in fold_scores}       # out-of-fold predictions: one per sentence
t0 = time.time()
for k, (tr_i, te_i) in enumerate(skf.split(txt_all, y_all), 1):
    tr_t, te_t = txt_all.iloc[tr_i], txt_all.iloc[te_i]
    ytr, yte = y_all[tr_i], y_all[te_i]
    feats = {}
    for rep in {r for r, _, _ in cfg}:
        if rep == "Word2Vec avg":
            feats[rep] = w2v_features(tr_t, te_t)
        else:
            v = VEC[rep](); feats[rep] = (v.fit_transform(tr_t), v.transform(te_t))
    for rep, clf_name, p in cfg:
        Xtr, Xte = feats[rep]
        pred = CLF[clf_name](p, rep).fit(Xtr, ytr).predict(Xte)
        fold_scores[(rep, clf_name)].append(metrics(yte, pred)); oof[(rep, clf_name)][te_i] = pred
    pred = DummyClassifier(strategy="most_frequent").fit(feats["BoW"][0], ytr).predict(feats["BoW"][1])
    fold_scores[("-", "Majority class")].append(metrics(yte, pred)); oof[("-", "Majority class")][te_i] = pred
    print(f"fold {k}/5 done  ({time.time() - t0:.0f}s)", flush=True)

### 7.1 Cross-validated results
Mean and standard deviation over the 5 folds. The last column is the macro-F1 computed once over **all 59,880 out-of-fold predictions**.

In [ ]:
rows = []
for (rep, clf_name), sc in fold_scores.items():
    s = pd.DataFrame(sc)
    pooled = metrics(y_all, oof[(rep, clf_name)])
    rows.append({"representation": rep, "classifier": clf_name,
                 "macro-F1 mean": s["macro_F1"].mean(), "macro-F1 std": s["macro_F1"].std(),
                 "accuracy mean": s["accuracy"].mean(), "macro-P mean": s["macro_P"].mean(), "macro-R mean": s["macro_R"].mean(),
                 "min fold F1": s["macro_F1"].min(), "max fold F1": s["macro_F1"].max(), "pooled macro-F1": pooled["macro_F1"]})
cv = pd.DataFrame(rows).sort_values("macro-F1 mean", ascending=False).reset_index(drop=True)
cv.index += 1
cv.style.format({c: "{:.4f}" for c in cv.columns[2:]}).background_gradient(subset=["macro-F1 mean"], cmap="Blues")

In [ ]:
plot = cv.copy()
plot["model"] = (plot["representation"].replace({"-": ""}) + " " + plot["classifier"]).str.strip()
plot = plot.sort_values("macro-F1 mean")
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(plot["model"], plot["macro-F1 mean"], xerr=plot["macro-F1 std"], color="#3b6ea5", capsize=3)
for i, (m, s) in enumerate(zip(plot["macro-F1 mean"], plot["macro-F1 std"])):
    ax.text(m + s + 0.004, i, f"{m:.3f}", va="center", fontsize=7)
ax.set_xlabel("macro-F1 (mean over 5 folds, bars = standard deviation)"); ax.set_title("Figure 8. Five-fold cross-validation")
plt.tight_layout(); plt.show()

# single split (Part 5) versus cross-validation
single = pd.DataFrame(results).set_index(["representation", "classifier"])["macro_F1"].rename("single-split test F1")
cmp_ = cv.set_index(["representation", "classifier"])[["macro-F1 mean", "macro-F1 std"]].join(single)
cmp_["difference"] = cmp_["single-split test F1"] - cmp_["macro-F1 mean"]
print("single split minus cross-validation mean (positive = the single split looked better):")
cmp_.drop(index=("-", "Majority class"), errors="ignore").sort_values("macro-F1 mean", ascending=False).round(4)

**Cross-validation interpretation:** *(write after you see the table: which model is best on average; whether the order of the models is the same as on the single split in Part 5; how large the standard deviation is compared with the gaps between models (a gap smaller than the standard deviation is not a reliable difference); whether any single-split score was far from its cross-validated mean)*